In [45]:
import os
import numpy as np
import trimesh
import tqdm 

os.makedirs('../results', exist_ok=True)

In [46]:
def uniform_sampling_from_mesh(vertices, faces, sample_num):
    # -------- TODO -----------
    # 1. compute area of each triangles
    # 2. compute probability of each triangles from areas
    # 3. sample N faces according to the probability
    # 4. for each face, sample 1 point
    # Note that FOR-LOOP is not allowed!
    # faces: (F, 3), vertices: (V, 3)
    tri = vertices[faces]  # (F, 3, 3)

    # 1) area of each triangle
    e1 = tri[:, 1, :] - tri[:, 0, :]
    e2 = tri[:, 2, :] - tri[:, 0, :]
    area = 0.5 * np.linalg.norm(np.cross(e1, e2), axis=1)  # (F,)

    # 2) probability from areas
    prob = area / (np.sum(area) + 1e-12)  # (F,)

    # 3) sample faces by probability
    face_idx = np.random.choice(faces.shape[0], size=sample_num, p=prob)
    tri_s = tri[face_idx]  # (N, 3, 3)

    # 4) uniformly sample one point inside each sampled triangle
    r1 = np.random.rand(sample_num, 1)
    r2 = np.random.rand(sample_num, 1)
    sr1 = np.sqrt(r1)

    w0 = 1.0 - sr1
    w1 = sr1 * (1.0 - r2)
    w2 = sr1 * r2

    uniform_pc = w0 * tri_s[:, 0, :] + w1 * tri_s[:, 1, :] + w2 * tri_s[:, 2, :]  # (N, 3)
    # -------- TODO -----------
    return area, prob, uniform_pc
        

In [47]:
def farthest_point_sampling(pc, sample_num):
    # -------- TODO -----------
    # FOR LOOP is allowed here.
    n = pc.shape[0]
    idx_list = np.zeros((sample_num,), dtype=np.int32)
    distances = np.full((n,), np.inf)
    farthest = np.random.randint(0, n)

    for i in range(sample_num):
        idx_list[i] = farthest
        dist = np.linalg.norm(pc - pc[farthest], axis=1)
        distances = np.minimum(distances, dist)
        farthest = np.argmax(distances)

    
    # -------- TODO -----------
    results = pc[idx_list]
    return results

In [48]:
# task 1: uniform sampling 

obj_path = 'spot.obj'
mesh = trimesh.load(obj_path)
print('faces shape: ', mesh.faces.shape)
sample_num = 512
area, prob, uniform_pc = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, sample_num)

# Visualization. For you to check your code
np.savetxt('uniform_sampling_vis.txt', uniform_pc)

print('area shape: ',area.shape)
print('prob shape: ',prob.shape)
print('pc shape: ',uniform_pc.shape)
# the result should satisfy: 
#       area.shape = (13712, ) 
#       prob.shape = (13712, ) 
#       uniform_pc.shape = (512, 3) 

# For submission
save_dict = {'area': area, 'prob': prob, 'pc': uniform_pc}
np.save('../results/uniform_sampling_results', save_dict)

faces shape:  (13712, 3)
area shape:  (13712,)
prob shape:  (13712,)
pc shape:  (512, 3)


In [49]:
# task 2: FPS

init_sample_num = 2000
final_sample_num = 512
_,_, tmp_pc = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, init_sample_num)
fps_pc = farthest_point_sampling(tmp_pc, final_sample_num)

# Visualization. For you to check your code
np.savetxt('fps_vis.txt', fps_pc)

# For submission
np.save('../results/fps_results', fps_pc)

In [61]:
# task 3: metrics


earthmover_distance = None

# If you want to use the reference implementation above, clone it under
# mesh_pc/earthmover first. You may also replace it with your own EMD code.
# -----------TODO---------------
# compute chamfer distance and EMD for two point clouds sampled by uniform sampling and FPS.
# sample and compute CD and EMD again. repeat for five times.
# save the mean and var.
from scipy.optimize import linear_sum_assignment

def chamfer_distance_mean(p1, p2):
    # mean NN distance in both directions
    d = np.linalg.norm(p1[:, None, :] - p2[None, :, :], axis=2)  # (N1, N2)
    return 0.5 * (d.min(axis=1).mean() + d.min(axis=0).mean())

def emd_distance_mean(p1, p2):
    # fallback: Hungarian exact matching (for equal-size sets)
    d = np.linalg.norm(p1[:, None, :] - p2[None, :, :], axis=2)
    r, c = linear_sum_assignment(d)
    return float(d[r, c].mean())

# repeat 5 times
CD_list = []
EMD_list = []

# make this cell independent
mesh = trimesh.load('spot.obj')
uniform_num = 512
init_sample_num = 2000
final_sample_num = 512

for _ in range(5):
    _, _, uniform_pc = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, uniform_num)
    _, _, tmp_pc = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, init_sample_num)
    fps_pc = farthest_point_sampling(tmp_pc, final_sample_num)

    CD_list.append(chamfer_distance_mean(uniform_pc, fps_pc))
    EMD_list.append(emd_distance_mean(uniform_pc, fps_pc))

CD_list = np.array(CD_list, dtype=np.float64)
EMD_list = np.array(EMD_list, dtype=np.float64)

CD_mean = float(CD_list.mean())
CD_var = float(CD_list.var())
EMD_mean = float(EMD_list.mean())
EMD_var = float(EMD_list.var())
# -----------TODO---------------

print(f"CD mean: {CD_mean}, CD var: {CD_var}")
print(f"EMD mean: {EMD_mean}, EMD var: {EMD_var}")

# For submission
np.save('../results/metrics', {'CD_mean':CD_mean, 'CD_var':CD_var, 'EMD_mean':EMD_mean, 'EMD_var':EMD_var})

CD mean: 1.370079621439898, CD var: 0.00026281561984328175
EMD mean: 2.509992539529798, EMD var: 0.01773976284089145
